In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gemma4:26b-a4b-it-qat"
API_LOCAL = "http://127.0.0.1:11434"

# Leer archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

# Verificar integridad
assert sha_casos == configuracion["cases_sha256"]

assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["prompt_version"] == "V3"
assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# Conectar con Ollama local
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as f:
    version_ollama = json.load(f)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Modelo no instalado."

# Consultar características reales
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as f:
    detalles = json.load(f)

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Parámetros:", detalles.get("parameters"))

print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: gemma4:26b-a4b-it-qat
Identificador: 2dd70431afed
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['completion', 'vision', 'tools', 'thinking']
Parámetros: temperature                    1
top_k                          64
top_p                          0.95

Archivos y conexión: correctos


In [2]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

ARCHIVO = "resultados.jsonl"
CASO_ID = "DIAG-G4-26B-A4B-01"

# 1. Caso diagnóstico común
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# 2. Recuperar configuración compartida V3
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO

# Desactivar explícitamente el razonamiento
solicitud["think"] = False

# Parámetros para Gemma 4 en modo directo
solicitud["options"] = {
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 1,
    "top_k": 64,
    "top_p": 0.95
}

# Mantener los prompts V3 originales
solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id=CASO_ID,
            contenido=contenido
        )
    }
]

# 3. Verificaciones
assert configuracion["prompt_version"] == "V3"
assert MODELO == "gemma4:26b-a4b-it-qat"
assert "thinking" in detalles.get("capabilities", [])
assert solicitud["think"] is False
assert sha_casos == configuracion["cases_sha256"]
assert modelo_info["digest"].startswith("2dd70431afed")

# Evitar duplicar el diagnóstico
try:
    with open(ARCHIVO, encoding="utf-8") as f:
        anteriores = [
            json.loads(line)
            for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r.get("case_id") == CASO_ID
    for r in anteriores
), "El diagnóstico ya está registrado."

# 4. Preparar registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "diagnostico",
    "configuracion_modelo": "gemma4_26b_a4b_qat_directa_v1",
    "prompt_version": "V3",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 5. Ejecutar diagnóstico en Ollama local
try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=900) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif mensaje.get("thinking"):
        registro["estado"] = "revisar_modo"

    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Guardar sin sobrescribir resultados
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 7. Mostrar resultados
print("Caso:", CASO_ID)
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    mensaje = r.get("message", {})

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        mensaje.get("thinking")
    ))

    for campo in (
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)

        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(mensaje.get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


Caso: DIAG-G4-26B-A4B-01
Estado: ok
Tiempo cliente: 33.78 s
Tokens entrada: 374
Tokens generados: 314
Terminación: stop
Razonamiento separado: False
total_duration 33.778 s
load_duration 17.946 s
prompt_eval_duration 8.09 s
eval_duration 7.734 s

RESPUESTA FINAL:

**Caso DIAG-G4-26B-A4B-01**

**Evidencia**
*   [E1] El Autor X posee una publicación centrada en bases de datos académicas.
*   [E2] El Autor Y posee una publicación orientada a la recuperación de información.
*   No se ha documentado la existencia de colaboraciones previas entre ambos autores.

**Interpretación**
La predicción de coautoría se basa en la convergencia temática de las áreas de investigación de ambos autores. El contenido de [E1] y [E2] sugiere una proximidad disciplinar, dado que las bases de datos académicas y la recuperación de información son campos estrechamente relacionados dentro de la ciencia de la información.

**Limitaciones**
La evidencia es insuficiente para establecer una relación de trabajo previa,

In [3]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gemma4:26b-a4b-it-qat"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"
CONFIG_MODELO = "gemma4_26b_a4b_qat_directa_v1"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]

assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["prompt_version"] == "V3"
assert config["api_url"] == API + "/api/chat"

# 2. Recuperar diagnóstico exitoso
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line)
        for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-G4-26B-A4B-01"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert diagnostico["solicitud"]["think"] is False

assert diagnostico["solicitud"]["options"] == {
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 1,
    "top_k": 64,
    "top_p": 0.95
}

assert diagnostico["sha256_casos"] == sha_casos
assert diagnostico["sha256_configuracion"] == sha_config

assert diagnostico["solicitud"]["messages"][0]["content"] == (
    config["system_prompt"]
)

# 3. Verificar identidad del modelo
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar duplicar ejecuciones
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == CONFIG_MODELO
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado", flush=True)
            continue

        # Recuperar solicitud exacta del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        # Sustituir únicamente el caso y sus evidencias
        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        # Preparar registro
        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": CONFIG_MODELO,
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        print("Ejecutando:", case_id, flush=True)
        inicio = time.perf_counter()

        # 5. Consultar API local
        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=900) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif mensaje.get("thinking"):
                registro["estado"] = "revisar_modo"

            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # 6. Guardar cada resultado inmediatamente
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s",
            flush=True
        )

print("\nArchivo actualizado:", ARCHIVO)


Ejecutando: S-005
S-005 | ok | 405 tokens | 16.159 s
Ejecutando: S-006
S-006 | ok | 294 tokens | 10.795 s
Ejecutando: S-007
S-007 | ok | 389 tokens | 11.142 s
Ejecutando: S-008
S-008 | ok | 293 tokens | 8.184 s
Ejecutando: S-009
S-009 | ok | 272 tokens | 7.407 s
Ejecutando: S-010
S-010 | ok | 332 tokens | 9.16 s

Archivo actualizado: resultados.jsonl


In [4]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gemma4:26b-a4b-it-qat"
ARCHIVO = "resultados.jsonl"
CONFIG_THINK = "gemma4_26b_a4b_qat_thinking_v1"
API = "http://127.0.0.1:11434"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

config = json.loads(bytes_config)
sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)
assert config["prompt_version"] == "V3"

# 2. Recuperar el diagnóstico original
with open(ARCHIVO, encoding="utf-8") as f:
    registros = [
        json.loads(line) for line in f if line.strip()
    ]

original = next(
    r for r in registros
    if r["case_id"] == "DIAG-G4-26B-A4B-01"
    and r["configuracion_modelo"] ==
        "gemma4_26b_a4b_qat_directa_v1"
)

assert original["estado"] == "ok"
assert original["solicitud"]["model"] == MODELO
assert original["solicitud"]["think"] is False
assert original["sha256_casos"] == sha_casos
assert original["sha256_configuracion"] == sha_config

# No repetir una configuración ya registrada
assert not any(
    r.get("case_id") == original["case_id"]
    and r.get("configuracion_modelo") == CONFIG_THINK
    for r in registros
), "El diagnóstico con razonamiento ya está registrado."

# 3. Conservar la misma entrada
solicitud = copy.deepcopy(original["solicitud"])

# Activar razonamiento y ampliar presupuesto de salida
solicitud["think"] = True
solicitud["options"]["num_predict"] = 3072

assert solicitud["options"]["num_ctx"] == 4096
assert solicitud["options"]["temperature"] == 1
assert solicitud["options"]["top_k"] == 64
assert solicitud["options"]["top_p"] == 0.95

# 4. Confirmar identidad del modelo
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_actual = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_actual["digest"] == original["modelo_digest"]

# 5. Preparar registro independiente
registro = {
    "execution_id": str(uuid4()),
    "case_id": original["case_id"],
    "grupo": "diagnostico",
    "configuracion_modelo": CONFIG_THINK,
    "referencia_execution_id": original["execution_id"],
    "prompt_version": "V3",
    "modelo_digest": original["modelo_digest"],
    "ollama_version": original["ollama_version"],
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 6. Ejecutar diagnóstico con razonamiento
try:
    peticion = Request(
        config["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=900) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})
    contenido_final = mensaje.get("content", "").strip()
    razonamiento = mensaje.get("thinking", "")

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"
    elif respuesta.get("done_reason") == "stop" and contenido_final:
        registro["estado"] = (
            "ok" if razonamiento else "revisar_modo"
        )
    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 7. Guardar sin alterar las siete ejecuciones previas
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 8. Mostrar métricas y respuesta final
print("Caso:", registro["case_id"])
print("Configuración:", CONFIG_THINK)
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    mensaje = r.get("message", {})

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        mensaje.get("thinking")
    ))

    for campo in (
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)
        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(mensaje.get("content", ""))
else:
    print("ERROR:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Caso: DIAG-G4-26B-A4B-01
Configuración: gemma4_26b_a4b_qat_thinking_v1
Estado: ok
Tiempo cliente: 35.591 s
Tokens entrada: 372
Tokens generados: 1298
Terminación: stop
Razonamiento separado: True
load_duration 0.007 s
prompt_eval_duration 0.599 s
eval_duration 34.894 s

RESPUESTA FINAL:

**Evidencia**
* [E1] El Autor X tiene un artículo sobre bases de datos académicas.
* [E2] El Autor Y tiene un artículo sobre recuperación de información.

**Interpretación**
Existe una convergencia temática entre los campos de estudio de ambos autores ([E1], [E2]), dado que la recuperación de información es un componente técnico fundamental en el ámbito de las bases de datos académicas. Esta relación temática sugiere que existe una pertinencia académica para una posible colaboración entre ambos investigadores.

**Limitaciones**
No se dispone de información sobre colaboraciones previas, publicaciones conjuntas, proyectos compartidos o afiliaciones institucionales comunes. La similitud temática evidencia

In [5]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gemma4:26b-a4b-it-qat"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"
CONFIG_THINK = "gemma4_26b_a4b_qat_thinking_v1"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["prompt_version"] == "V3"
assert config["api_url"] == API + "/api/chat"

# 2. Recuperar diagnóstico con razonamiento
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line)
        for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-G4-26B-A4B-01"
    and r["configuracion_modelo"] == CONFIG_THINK
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert diagnostico["solicitud"]["think"] is True

assert diagnostico["solicitud"]["options"] == {
    "num_ctx": 4096,
    "num_predict": 3072,
    "temperature": 1,
    "top_k": 64,
    "top_p": 0.95
}

assert diagnostico["sha256_casos"] == sha_casos
assert diagnostico["sha256_configuracion"] == sha_config

assert diagnostico["solicitud"]["messages"][0]["content"] == (
    config["system_prompt"]
)

# 3. Verificar identidad del modelo
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# No duplicar los casos registrados
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == CONFIG_THINK
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado", flush=True)
            continue

        # Reutilizar la solicitud del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        # Sustituir únicamente el caso
        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": CONFIG_THINK,
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        print("Ejecutando:", case_id, flush=True)
        inicio = time.perf_counter()

        # 5. Consultar Ollama
        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=900) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})
            contenido_final = mensaje.get("content", "").strip()
            razonamiento = mensaje.get("thinking", "")

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif respuesta.get("done_reason") == "stop" and contenido_final:
                registro["estado"] = (
                    "ok" if razonamiento else "revisar_modo"
                )

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # 6. Guardar inmediatamente cada resultado
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})
        mensaje = r.get("message", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s",
            "| thinking:", bool(mensaje.get("thinking")),
            flush=True
        )

print("\nArchivo actualizado:", ARCHIVO)


Ejecutando: S-005
S-005 | ok | 1208 tokens | 30.485 s | thinking: True
Ejecutando: S-006
S-006 | ok | 1278 tokens | 30.676 s | thinking: True
Ejecutando: S-007
S-007 | ok | 1914 tokens | 46.086 s | thinking: True
Ejecutando: S-008
S-008 | ok | 1398 tokens | 33.616 s | thinking: True
Ejecutando: S-009
S-009 | ok | 1732 tokens | 41.627 s | thinking: True
Ejecutando: S-010
S-010 | ok | 1029 tokens | 24.897 s | thinking: True

Archivo actualizado: resultados.jsonl
